In [1]:
import pandas as pd


data = pd.read_csv(
"../data/processed/aml_features.csv"
)


data.head()

,txId,time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,class,transaction_velocity,night_transaction,high_amount_flag
0,0,-1.556878,0.307025,1.168821,-0.748109,7.643807,-0.053406,6.039024,21.528480,-0.146334,...,2.115930,0.016322,-0.117599,0.593442,-0.08332,-0.082052,2,2147,0,0
1,1,-1.556878,0.067780,0.271911,-0.226081,2.628539,-0.053406,2.820415,1.501088,-0.146334,...,1.200416,-0.028141,-0.117599,0.270020,-0.08332,-0.082052,2,2147,0,0
2,2,-1.556878,-0.135517,-0.222646,-1.270138,-0.183761,-0.041930,-0.187137,-0.080022,-0.102664,...,0.056025,-0.083719,-0.117599,-0.134257,-0.08332,-0.082052,2,2147,0,0
3,3,-1.556878,-0.140505,-0.222646,-1.270138,-0.183761,-0.041930,-0.187137,-0.080022,-0.108753,...,0.056025,-0.083719,-0.117599,-0.134257,-0.08332,-0.082052,2,2147,0,0
4,4,-1.556878,-0.170324,-0.222646,-1.270138,-0.090018,-0.041930,-0.134373,0.447015,-0.146324,...,0.056025,-0.072604,-0.091257,-0.093830,-1.72568,-1.726407,2,2147,0,0


In [4]:
from pathlib import Path
import sys

# Locate the project root from the notebook's working directory
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src").is_dir()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing 'src'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.fraud_model import train_anomaly_model

In [6]:
# Exclude the transaction identifier and supervised target column
X = data.drop(columns=["txId", "class"])

model = train_anomaly_model(X)

In [7]:
from src.fraud_model import predict_anomaly


data["anomaly"] = predict_anomaly(
    model,
    X
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_864\856142514.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["anomaly"] = predict_anomaly(


In [8]:
data["is_suspicious"] = (
    data["anomaly"] == -1
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_864\1323197038.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["is_suspicious"] = (


In [9]:
from src.risk_engine import calculate_risk_score
from src.risk_engine import risk_category

In [10]:
data["risk_score"] = data.apply(
    calculate_risk_score,
    axis=1
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_864\1562066407.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["risk_score"] = data.apply(


In [11]:
data["risk_level"] = data["risk_score"].apply(
    risk_category
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_864\2854555425.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["risk_level"] = data["risk_score"].apply(


In [14]:
data[
[
"risk_score",
"risk_level",
"is_suspicious"
]
].head(10)

,risk_score,risk_level,is_suspicious
0,30,LOW,False
1,30,LOW,False
2,30,LOW,False
3,30,LOW,False
4,30,LOW,False
5,30,LOW,False
6,30,LOW,False
7,30,LOW,False
8,30,LOW,False
9,30,LOW,False


In [16]:
data.to_csv(
"../data/processed/fraud_detection_results.csv",
index=False
)